# WP05 — Tripoli retained-map GeoTIFF export

**حامد: این نوت‌بوک همان ZIP تکمیل‌شدهٔ WP04 را می‌گیرد و نقشهٔ نهایی را به Google Drive صادر می‌کند.**
کد مرجع GEE و روش مدل تغییر نمی‌کنند. جست‌وجوی seed تکرار نمی‌شود؛ سرور ممکن است گراف مدل انتخاب‌شده را برای تولید پیکسل‌ها محاسبه کند.

Run cells in order in a **fresh Colab runtime**. Cell 5 submits **one final-map export** to your account. Read its settings first. This notebook is distributed unexecuted; it does not claim an export has completed. Keep the original ZIP and download the backup before closing Colab.

[WP05 instructions](https://github.com/hamedsabzchi/libya-automated-land-cover-mapping/blob/feat/wp05-jobs-geotiff/docs/wp05-jobs-exports.md)


## 1. Install the pinned export service

In [ ]:
import subprocess, sys
WP05_COMMIT = "422283520f292f9302f53874f96a5741bc438a85"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
    "git+https://github.com/hamedsabzchi/libya-automated-land-cover-mapping.git@" + WP05_COMMIT])
import ee, libya_landcover
assert ee.__version__ == "1.7.48"
assert libya_landcover.__version__ == "0.5.0"
print("Export service:", libya_landcover.__version__, WP05_COMMIT)


## 2. Authenticate using your Google account
Use the account with access to the Tripoli assets and Google Drive. Credentials stay in this runtime; the backup below excludes them.

In [ ]:
PROJECT = "practical-proxy-441422-n6"
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine initialized:", PROJECT)


## 3. Upload and import your completed WP04 ZIP
Upload `wp04-tripoli-...zip` from the previous run. This imports checked JSON and image graphs; it does not execute code from the ZIP. The active store persists within this runtime only.

In [ ]:
import json, hashlib
from pathlib import Path
from uuid import uuid5, NAMESPACE_URL
from google.colab import files
from libya_landcover.job_store import JobStore, encoded
from libya_landcover.jobs import import_bundle, checked_mapping_bundle
from libya_landcover.exports import enqueue_export, submit_export, poll_export, export_spec

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one completed WP04 ZIP")
archive_name, archive_bytes = next(iter(uploaded.items()))
archive = Path("/content/wp05-input.zip")
archive.write_bytes(archive_bytes)
del uploaded, archive_bytes
store = JobStore("/content/wp05-job-store")
mapping = import_bundle(store, archive)
checked = checked_mapping_bundle(store, mapping)
result = checked["result"].to_dict()
assert checked["manifest"]["project"] == PROJECT
assert result["request"]["aoi_asset"] == "projects/practical-proxy-441422-n6/assets/Libya/Tripoli"
assert result["request"]["training_asset"] == "projects/practical-proxy-441422-n6/assets/Libya/TD3-attept1-tripoli"
print("Mapping:", mapping["job_id"], mapping["state"])
print("Source run:", result["run_id"])
print("Engine commit:", result["provenance"]["engine_commit"])
print(json.dumps(result["outputs"], indent=2))


## 4. Select one retained map and prepare its export
Default: the final temporal map, EPSG:32633, 10 m grid used in the recorded Tripoli probes. To export an annual map, explicitly set `MAP_KIND = "annual_map"` and `MAP_YEAR` to a listed year. Each selection is a separate task.

The request ID is deterministic: repeating this cell with the same ZIP, artifact and settings reuses the logical export identity, including after a runtime reset. Use the same Google account/project. **Do not change the UUID to work around an uncertain submission.** A deliberate new attempt after confirmed failure/cancellation requires a separately chosen new UUID.


In [ ]:
MAP_KIND = "final_map"
MAP_YEAR = None  # Required only when selecting an annual_map, e.g. 2024.
settings = export_spec({
    "crs": "EPSG:32633", "transform": [10, 0, 0, 0, -10, 0],
    "folder_prefix": "LibyaLandCover_Tripoli", "max_pixels": 100000000,
    "file_dimensions": 4096,
})
outputs = [item for item in result["outputs"] if item["kind"] == MAP_KIND
           and (MAP_YEAR is None or item["year"] == MAP_YEAR)]
if len(outputs) != 1:
    raise ValueError("Select exactly one output; specify a year for annual maps")
output = outputs[0]
index = json.loads(checked["files"]["graphs/index.json"])
identity = {"project": PROJECT, "run_id": result["run_id"],
    "artifact_id": output["artifact_id"],
    "graph_sha256": index["artifacts"][output["artifact_id"]]["sha256"], "spec": settings}
export_request_id = str(uuid5(NAMESPACE_URL, "libya-wp05-export:" + encoded(identity)))
export = enqueue_export(store, mapping["job_id"], output["artifact_id"],
                        settings, request_id=export_request_id)
EXPORT_JOB_ID = export["job_id"]
print(json.dumps({"job_id": EXPORT_JOB_ID, "state": export["state"],
    "ee_request_id": export["ee_request_id"], "selected_output": output,
    "project": PROJECT, "export_spec": settings}, indent=2))
print("Local preparation only; the next cell submits the task.")


## 5. Submit the export
Running this cell authorizes the selected task in your Earth Engine project and Drive. Repeating it uses the same request identity. A lost response can produce `submission_unknown`; use the poll cell to reconcile before deciding anything else. It may take time for a task to finish.


In [ ]:
export = submit_export(store, EXPORT_JOB_ID)
print(json.dumps(export, indent=2))


## 6. Refresh status — repeat this cell until terminal
`submitted`, `running`, `cancelling`, and `submission_unknown` are **not** completed exports. A polling transport error does not mean the server task failed. If an uncertain task cannot be reconciled, inspect Earth Engine Tasks; do not create a new request ID blindly.


In [ ]:
export = poll_export(store, EXPORT_JOB_ID)
print(json.dumps({key: export.get(key) for key in (
    "job_id", "state", "remote_state", "operation_name", "drive_folder",
    "filename_prefix", "destination_uris", "error")}, indent=2))
if export["state"] == "completed":
    print("Export completed. Open the service-provided Drive destination(s):")
    for uri in export["destination_uris"]:
        print(uri)
    print("If no URI is returned, find this exact folder in your Google Drive:", export.get("drive_folder"))
    print("Download all .tif parts. The backup below contains metadata, not raster pixels.")
elif export["state"] in {"failed", "cancelled"}:
    print("Terminal state; download the diagnostic backup below before any new attempt.")
else:
    print("Not completed yet. Repeat this polling cell later.")


## 7. Download manifest and a scoped recovery backup
Run this even while pending or after a failure. Repeat after the final status update. The ZIP includes this job store's database, retained graphs, export payload and an `export-manifest.json`; it excludes credentials and other runtime files. It does **not** contain GeoTIFF pixels. Keep it private.

Download TIFF files from the recorded Drive destination after success. To finish live acceptance, provide the manifest and a downloaded TIFF for checking grid/encoding, without reassessing model accuracy. If the output has multiple parts, download them all.


In [ ]:
import tempfile, shutil, zipfile
from datetime import datetime, timezone
from libya_landcover.bundle import write_json

export = store.get(EXPORT_JOB_ID)
manifest = {"schema_version": "wp05-export-handoff.1", "export_service_commit": WP05_COMMIT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "mapping": store.get(mapping["job_id"]), "export": export,
    "export_events": store.events(EXPORT_JOB_ID),
    "source_engine_commit": result["provenance"]["engine_commit"],
    "raster_included": False}
# No local worker or submission may run concurrently while this snapshot is copied.
with tempfile.TemporaryDirectory(prefix="wp05-backup-") as temporary:
    backup = Path(temporary)
    write_json(backup / "export-manifest.json", manifest)
    store.snapshot(backup / "jobs.sqlite3")
    shutil.copytree(store.root / "jobs", backup / "jobs")
    archive_path = Path("/content/wp05-export-" + EXPORT_JOB_ID + "-" + str(export["revision"]) + ".zip")
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as target:
        for path in sorted(backup.rglob("*")):
            if path.is_file():
                target.write(path, path.relative_to(backup))
files.download(str(archive_path))
print("Saved backup for state:", export["state"])


### Optional cancellation and recovery
Cancellation is not part of Run all. To request it, run `cancel_export(store, EXPORT_JOB_ID)` after importing that function from `libya_landcover.exports`, then poll until the server confirms cancellation or a racing completion.

For local recovery, restore a **trusted** backup to a fresh directory with `jobs.sqlite3` and `jobs/` together, construct `JobStore` for that directory, and poll the saved export job ID. Do not import a WP05 backup using the WP04 importer. Alternatively, re-upload the original WP04 ZIP and repeat cells 1–6 with identical settings/account; the deterministic request ID is preserved even though local job IDs change. A duplicate submission response may require polling to recover the operation. If no operation can be found, inspect Tasks and Drive before starting any new attempt.
